In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score



In [ ]:
!pip install geopy

In [ ]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="crop_app")

# lat/lon → district
def get_district(lat, lon):
    location = geolocator.reverse((lat, lon), language='en')

    if location is None:
        return None

    address = location.raw.get('address', {})

    district = (
        address.get('state_district') or
        address.get('county') or
        address.get('district') or
        address.get('city')
    )

    if district:
        return district.lower().replace(" district", "").strip()

    return None


# district → lat/lon
def get_lat_lon_from_district(district):
    location = geolocator.geocode(f"{district}, India")

    if location:
        return location.latitude, location.longitude
    return None, None

In [ ]:
# Load Kaggle dataset
df = pd.read_csv("Crop_Recommendation_dataset.csv")

X = df[['N','P','K','temperature','humidity','ph','rainfall']]
y = df['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Accuracy
y_pred = model.predict(X_test)
print("Top-1:", accuracy_score(y_test, y_pred))

y_probs = model.predict_proba(X_test)
print("Top-3:", top_k_accuracy_score(y_test, y_probs, k=3))

# Save
joblib.dump(model, "rf_model.pkl")
joblib.dump(le, "label_encoder.pkl")

Top-1: 0.9931818181818182
Top-3: 1.0


['label_encoder.pkl']

In [ ]:
hist_df = pd.read_csv("final_training_dataset.csv")

hist_df.columns = hist_df.columns.str.strip().str.lower()

hist_df['crop'] = hist_df['crop'].str.lower()
hist_df['district'] = hist_df['district'].str.lower()
hist_df['season'] = hist_df['season'].str.lower()

hist_df['yield'] = pd.to_numeric(hist_df['yield'], errors='coerce')
hist_df['area'] = pd.to_numeric(hist_df['area'], errors='coerce')

In [ ]:
district_crop_map = {}

for _, row in hist_df.iterrows():
    d, s, c = row['district'], row['season'], row['crop']

    district_crop_map.setdefault(d, {})
    district_crop_map[d].setdefault(s, set())
    district_crop_map[d][s].add(c)

for d in district_crop_map:
    for s in district_crop_map[d]:
        district_crop_map[d][s] = list(district_crop_map[d][s])

In [ ]:
max_yield = hist_df.groupby(['district','season'])['yield'].transform('max')
hist_df['yield_score'] = hist_df['yield'] / max_yield

total_area = hist_df.groupby(['district','season'])['area'].transform('sum')
hist_df['area_share'] = hist_df['area'] / total_area

In [ ]:
historical_lookup = {}

for _, row in hist_df.iterrows():
    d, s, c = row['district'], row['season'], row['crop']

    historical_lookup.setdefault(d, {})
    historical_lookup[d].setdefault(s, {})

    historical_lookup[d][s][c] = {
        "yield_score": row['yield_score'],
        "area_share": row['area_share']
    }

In [ ]:
def recommend_crop(district, season, soil_weather):

    district = district.strip().lower()
    season = season.strip().lower()

    if district not in district_crop_map:
        return []

    if season not in district_crop_map[district]:
        return []

    allowed_crops = district_crop_map[district][season]

    input_df = pd.DataFrame([{
        'N': soil_weather[0],
        'P': soil_weather[1],
        'K': soil_weather[2],
        'temperature': soil_weather[3],
        'humidity': soil_weather[4],
        'ph': soil_weather[5],
        'rainfall': soil_weather[6]
    }])

    probs = model.predict_proba(input_df)[0]
    prob_dict = dict(zip(le.classes_, probs))

    results = []

    for crop in allowed_crops:

        crop = crop.lower()

        agro = prob_dict.get(crop, prob_dict.get('maize', 0)*0.85)

        yield_s = historical_lookup.get(district, {}) \
            .get(season, {}) \
            .get(crop, {}) \
            .get("yield_score", 0)

        area_s = historical_lookup.get(district, {}) \
            .get(season, {}) \
            .get(crop, {}) \
            .get("area_share", 0)

        final_score = 0.5*agro + 0.3*yield_s + 0.2*area_s

        results.append((crop, final_score))

    return sorted(results, key=lambda x: x[1], reverse=True)

In [ ]:
print(recommend_crop("anantapur", "kharif", [90,40,40,25,70,6.5,200])[:3])

[('maize', np.float64(0.315968992248062)), ('rice', np.float64(0.3025765914534044)), ('cereals', np.float64(0.2306144967124722))]


In [ ]:
district_correct_top1 = 0
district_correct_top3 = 0
district_total = 0

for district in district_crop_map:
    for season in district_crop_map[district]:

        soil_weather = [90, 40, 40, 25, 70, 6.5, 200]

        try:
            predictions = recommend_crop(
                district, season, soil_weather
            )

            district_data = hist_df[
                (hist_df['district'] == district) &
                (hist_df['season'] == season)
            ]

            if len(district_data) == 0:
                continue

            top_hist_crop = district_data.sort_values(
                by='yield', ascending=False
            ).iloc[0]['crop']

            # Top-1
            if predictions[0][0] == top_hist_crop:
                district_correct_top1 += 1

            # Top-3
            top3 = [p[0] for p in predictions[:3]]
            if top_hist_crop in top3:
                district_correct_top3 += 1

            district_total += 1

        except:
            continue

print("District-Level Top-1 Match:",
      district_correct_top1 / district_total)

print("District-Level Top-3 Match:",
      district_correct_top3 / district_total)


District-Level Top-1 Match: 0.5966149506346967
District-Level Top-3 Match: 0.8913963328631875


In [ ]:
import os
os.makedirs("final_model", exist_ok=True)

In [ ]:
import joblib

joblib.dump(model, "final_model/rf_model.pkl")
joblib.dump(le, "final_model/label_encoder.pkl")

['final_model/label_encoder.pkl']

In [ ]:
import json

def convert_sets(obj):
    if isinstance(obj, set):
        return list(obj)
    return obj

with open("final_model/district_crop_map.json", "w") as f:
    json.dump(district_crop_map, f, default=convert_sets)

In [ ]:
with open("final_model/historical_lookup.json", "w") as f:
    json.dump(historical_lookup, f)

In [ ]:
config = {
    "agro_weight": 0.5,
    "yield_weight": 0.3,
    "area_weight": 0.2
}

with open("final_model/config.json", "w") as f:
    json.dump(config, f)

In [ ]:
!pip install earthengine-api requests joblib pandas numpy

In [ ]:
import ee
import requests
import pandas as pd
import numpy as np
import joblib
import json

In [ ]:
ee.Authenticate()
ee.Initialize(project='crop-suitability-project')

In [ ]:
def get_season_dates(season):

    season = season.lower()

    if season == "kharif":
        return '2025-06-01', '2025-10-31'

    elif season == "rabi":
        return '2025-10-01', '2026-03-31'

    else:
        # fallback
        return '2025-01-01', '2025-12-31'

In [ ]:
def mask_s2_clouds(img):
    # S2 QA60 band: cloud bits
    qa = img.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return img.updateMask(mask)

In [ ]:
def get_ndvi(lat, lon, season):

    start, end = get_season_dates(season)
    region = ee.Geometry.Point([lon, lat]).buffer(500)

    col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(region)
           .filterDate(start, end)
           .map(mask_s2_clouds))

    # Compute NDVI for each image
    def add_ndvi(img):
        nd = img.normalizedDifference(['B8', 'B4']).rename('NDVI')
        return img.addBands(nd)

    col = col.map(add_ndvi)

    # 🔥 Take MAX NDVI over season (peak crop stage)
    ndvi_max = col.select('NDVI').max()

    stats = ndvi_max.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=10,
        bestEffort=True
    ).getInfo()

    return stats.get('NDVI', 0.3)  # safe fallback

In [ ]:
def land_suitability(lat, lon, season):

    ndvi = get_ndvi(lat, lon, season)

    if ndvi > 0.5:
        return "Highly Suitable", ndvi
    elif ndvi > 0.25:
        return "Moderately Suitable", ndvi
    else:
        return "Low Suitable", ndvi

In [ ]:
def get_weather(lat, lon):

    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true&hourly=relativehumidity_2m,precipitation"

    data = requests.get(url).json()

    temp = data['current_weather']['temperature']
    current_time = data['current_weather']['time']

    hourly_times = data['hourly']['time']

    # 🔥 FIX: find closest time instead of exact match
    idx = min(
        range(len(hourly_times)),
        key=lambda i: abs(pd.to_datetime(hourly_times[i]) - pd.to_datetime(current_time))
    )

    humidity = data['hourly']['relativehumidity_2m'][idx]
    rainfall = data['hourly']['precipitation'][idx]

    return temp, humidity, rainfall

In [ ]:
model = joblib.load("final_model/rf_model.pkl")
le = joblib.load("final_model/label_encoder.pkl")

with open("final_model/district_crop_map.json") as f:
    district_crop_map = json.load(f)

with open("final_model/historical_lookup.json") as f:
    historical_lookup = json.load(f)

In [ ]:
def full_system(
    lat=None,
    lon=None,
    district=None,
    season="kharif",
    mode="auto",
    N=None,
    P=None,
    K=None
):

    # 🔥 LOCATION HANDLING

    if mode == "auto":
        # user gives district → we auto get lat/lon
        lat, lon = get_lat_lon_from_district(district)
        district = district.lower()

    elif mode == "coords":
        # user gives lat/lon → we auto detect district
        district = get_district(lat, lon)

    if lat is None or lon is None or district is None:
        return {"error": "Invalid location input"}

    print("Using District:", district)

    # 🔥 NPK HANDLING (NEW FEATURE)
    if N is None: N = 90
    if P is None: P = 40
    if K is None: K = 40

    # 1. NDVI (DO NOT BLOCK)
    suitability, ndvi = land_suitability(lat, lon, season)

    # 2. Weather
    try:
        temp, humidity, rainfall = get_weather(lat, lon)
    except:
        temp, humidity, rainfall = 25, 70, 200

    soil_weather = [N, P, K, temp, humidity, 6.5, rainfall]

    # 3. ML prediction
    input_df = pd.DataFrame([{
        'N': N,
        'P': P,
        'K': K,
        'temperature': temp,
        'humidity': humidity,
        'ph': 6.5,
        'rainfall': rainfall
    }])

    probs = model.predict_proba(input_df)[0]
    prob_dict = dict(zip(le.classes_, probs))

    # 🔥 FILTER WITH FALLBACK
    if district not in district_crop_map or season not in district_crop_map[district]:

        sorted_crops = sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)

        return {
            "Suitability": suitability,
            "NDVI": round(ndvi, 2),
            "Top Crops": sorted_crops[:3]
        }

    allowed = district_crop_map[district][season]

    results = []

    for crop in allowed:

        crop = crop.lower()

        agro = prob_dict.get(crop, prob_dict.get('maize', 0)*0.85)

        yield_s = historical_lookup.get(district, {}) \
            .get(season, {}) \
            .get(crop, {}) \
            .get("yield_score", 0)

        area_s = historical_lookup.get(district, {}) \
            .get(season, {}) \
            .get(crop, {}) \
            .get("area_share", 0)

        final_score = 0.5*agro + 0.3*yield_s + 0.2*area_s

        results.append((crop, final_score))

    results.sort(key=lambda x: x[1], reverse=True)

    return {
        "Suitability": suitability,
        "NDVI": round(ndvi, 2),
        "Top Crops": results[:3]
    }

In [ ]:
print(full_system(
    district="koppal",
    season="kharif",
    mode="auto"
))

Using District: koppal
{'Suitability': 'Highly Suitable', 'NDVI': 0.6, 'Top Crops': [('rice', np.float64(0.31005110732538327)), ('cereals', np.float64(0.2629955400370159)), ('maize', np.float64(0.23749344981204973))]}


In [ ]:
print(full_system(
    district="koppal",
    season="kharif",
    mode="auto",
    N=120,
    P=60,
    K=50
))

Using District: koppal
{'Suitability': 'Highly Suitable', 'NDVI': 0.6, 'Top Crops': [('rice', np.float64(0.3117177739920499)), ('cereals', np.float64(0.2587455400370159)), ('maize', np.float64(0.23249344981204972))]}


In [ ]:
print(full_system(
    lat=15.64232,
    lon=76.60902,
    season="kharif",
    mode="coords"
))

Using District: koppal
{'Suitability': 'Highly Suitable', 'NDVI': 0.63, 'Top Crops': [('rice', np.float64(0.31005110732538327)), ('cereals', np.float64(0.25166220670368256)), ('maize', np.float64(0.2241601164787164))]}
